<center><img src="img/header_gr13_outlier_anomalies.jpg"></center>

# Airbnb Listings
The dataset contains information on **17,432 Airbnb listings in Bangkok**. Each row represents a listing with details such as coordinates, neighborhood, host id, price per night, number of reviews, and so on. 

[Source: InsideAirbnb](http://insideairbnb.com)

## Data Dictionary

| Column                            | Explanation                                                                                                                                                                                        |
| --------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| id                                | Airbnb's unique identifier for the listing                                                                                                                                                         |
| name                              |                                                                                                                                                                                                    |
| host\_id                          |                                                                                                                                                                                                    |
| host\_name                        |                                                                                                                                                                                                    |
| neighbourhood\_group              | The neighbourhood group as geocoded using the latitude and longitude against neighborhoods as defined by open or public digital shapefiles.                                                        |
| neighbourhood                     | The neighbourhood as geocoded using the latitude and longitude against neighborhoods as defined by open or public digital shapefiles.                                                              |
| latitude                          | Uses the World Geodetic System (WGS84) projection for latitude and longitude.                                                                                                                      |
| longitude                         | Uses the World Geodetic System (WGS84) projection for latitude and longitude.                                                                                                                      |
| room\_type                        |                                                                                                                                                                                                    |
| price                             | daily price in local currency. Note, $ sign may be used despite locale                                                                                                                             |
| minimum\_nights                   | minimum number of night stay for the listing (calendar rules may be different)                                                                                                                     |
| number\_of\_reviews               | The number of reviews the listing has                                                                                                                                                              |
| last\_review                      | The date of the last/newest review                                                                                                                                                                 |
| calculated\_host\_listings\_count | The number of listings the host has in the current scrape, in the city/region geography.                                                                                                           |
| availability\_365                 | avaliability\_x. The availability of the listing x days in the future as determined by the calendar. Note a listing may be available because it has been booked by a guest or blocked by the host. |
| number\_of\_reviews\_ltm          | The number of reviews the listing has (in the last 12 months)                                                                                                                                      |
| license                           |                                                                                                                                                                                                    |

The data for each city was compiled by [InsideAirbnb](http://insideairbnb.com) between October and November 2021.

[Source](http://insideairbnb.com/get-the-data.html) and [license](https://creativecommons.org/licenses/by/4.0/) of dataset. 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore
from pyod.models.mad import MAD


df = pd.read_csv("listings_bangkok.csv")

print(list(df.columns))
prices = df['price']

# Print 5-number summary
print(prices.describe())

<h1 style="font-weight: bold;"> <center>HISTOGRAM</center></h1> 
A Histogram is a graph that displays the frequency distribution of a single numerical variable by grouping data into "bins" (ranges).

**Role in Outlier Analysis:** It allows you to see the "shape" of your data. If most listings are priced between $\$500$ and $\$5,000$, but you see a tiny, isolated bar far to the right at $\$50,000$, that indicates a potential outlier.

In [ ]:
# Find the square root of the length of prices
n_bins = np.sqrt(len(prices))/6

# Cast to an integer
n_bins = int(n_bins)

plt.figure(figsize=(10, 6))

# Added edge color to distinguish between bins
plt.hist(prices, bins=n_bins, color='red', edgecolor='black')

plt.title('Distribution of Bangkok Airbnb Prices')
plt.xlabel('Price (THB)')
plt.ylabel('Frequency (Number of Listings)')

# Optional: Limit the x-axis if outliers hide the main distribution
# plt.xlim(0, prices.quantile(0.95)) 

plt.grid(axis='y', alpha=0.75)
plt.show()

<h1 style="font-weight: bold;"> <center>BOX PLOTS</center></h1> 

A Box Plot (or box-and-whisker plot) is a visual representation of the "Five-Number Summary": Minimum, First Quartile ($Q_{1}$), Median, Third Quartile ($Q_{3}$), and Maximum.

**Role in Outlier Analysis:** It is specifically designed to highlight outliers. Points that fall outside the "whiskers" are typically plotted individually as dots, signaling they are statistically distant from the rest of the data.

In [ ]:
plt.boxplot(prices, whis=1.5, tick_labels=['Bangkok Listings'])

# Add the labels
plt.title("Distribution of Prices in Bangkok Listings")
plt.xlabel("Data Category")
plt.ylabel("Price (in THB or local currency)")

# Add a grid (Optional but helpful for reading values)
plt.grid(axis='y', linestyle='--', alpha=0.7)
# plt.ylim(0, 20000)

plt.show()

<h1 style="font-weight: bold;"> <center>Interquartile Range(IQR)</center></h1> 

The Interquartile Range (IQR) is a measure of statistical dispersion, calculated as the difference between the 75th percentile ($Q_{3}$) and the 25th percentile ($Q_{1}$). We use it to create "fences." It is highly robust (meaning it isn't easily confused by a few massive errors), making it a reliable tool for cleaning Bangkok price data where extreme values are common.

In [ ]:
# Calculate the 25th and 75th percentiles
q1 = prices.quantile(0.25)
q3 = prices.quantile(0.75)

# Find the IQR
IQR = q3 - q1
factor = 2.5

# Calculate the lower limit
lower_limit = q1 - (IQR * factor)

# Calculate the upper limit
upper_limit = q3 + (IQR * factor)

# Create a mask for values lower than lower_limit
is_lower = prices < lower_limit

# Create a mask for values higher than upper_limit
is_higher = prices > upper_limit

# Combine the masks to filter for outliers
outliers = prices[is_lower | is_higher]

# Count and print the number of outliers
print(f"Total outliers found: {len(outliers)}")

<h1 style="font-weight: bold;"> <center>Z-SCORE</center></h1> 

The Z-Score tells us how many standard deviations a price is from the average. A score of $3$ means the price is very far from the "norm." While common, this tool is sensitive. If we have one listing at $1,000,000$ THB, it will pull the average up and might make other real outliers look "normal."

In [ ]:

# Find the zscores of prices
scores = zscore(prices)

# Check if the absolute values of scores are over 3
is_over_3 = np.abs(scores) > 3

# Use the mask to subset prices
outliers = scores[is_over_3]

print(f"Total outliers found: {len(outliers)}")

<h1 style="font-weight: bold;"> <center>Median Absolute Deviation(MAD)</center></h1> 

MAD is the "strictest" univariate tool. Instead of using the average (which outliers can ruin), it uses the Median. In the context of the Bangkok Airbnb market, where you have a huge variety of prices, MAD is often the most realistic tool for finding anomalies because it stays "centered" on the most common prices.

In [ ]:
# 1. Initialize the model
mad = MAD(threshold=3.5)

# 2. Reshape your data
prices_reshaped = prices.values.reshape(-1, 1)

# 3. Fit the model to the data first
mad.fit(prices_reshaped)

# 4. Then predict the labels (this replaces fit_predict)
# In many outlier libraries, labels are stored in the 'labels_' attribute after fitting
labels = mad.labels_ 

# 5. Filter for outliers (where label 1 is usually an outlier in PyOD)
outliers = prices[labels == 1]

print(f"Total outliers found: {len(outliers)}")